# Notebook 05 — Downstream Biological Analysis

This notebook asks: **do different segmentation methods lead to different biological conclusions?**

For each method we:
1. Build a cell × gene count matrix from transcript assignments
2. Run standard scRNA-seq analysis (normalisation → PCA → UMAP → Leiden clustering)
3. Identify marker genes per cluster
4. Compare cell-type proportions and spatial organisation across methods

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
import squidpy as sq
import anndata as ad
from pathlib import Path
from scipy.sparse import csr_matrix

sc.settings.verbosity = 1
sc.settings.figdir = "../results/"

DATA_DIR = Path("../data")
RESULTS_DIR = Path("../results")

METHODS = ["xenium", "cellpose", "baysor"]
TRANSCRIPT_FILES = {
    "xenium": DATA_DIR / "transcripts_filtered.parquet",
    "cellpose": DATA_DIR / "transcripts_cellpose.parquet",
    "baysor": DATA_DIR / "transcripts_baysor.parquet",
}
CELL_ID_COLS = {
    "xenium": "cell_id",
    "cellpose": "cellpose_cell_id",
    "baysor": "cell",
}

## 1. Build Cell × Gene Count Matrices

In [ ]:
def build_count_matrix(transcripts_df: pd.DataFrame, cell_col: str, gene_col: str = "feature_name") -> ad.AnnData:
    """Pivot transcript-level data into a cells × genes AnnData."""
    assigned = transcripts_df[
        transcripts_df[cell_col].notna() & (transcripts_df[cell_col] != 0) & (transcripts_df[cell_col] != -1)
    ].copy()
    assigned[cell_col] = assigned[cell_col].astype(str)

    counts = assigned.groupby([cell_col, gene_col]).size().unstack(fill_value=0)
    adata = ad.AnnData(X=csr_matrix(counts.values))
    adata.obs_names = counts.index.astype(str)
    adata.var_names = counts.columns

    # Attach centroid coordinates
    centroids = assigned.groupby(cell_col)[["x_location", "y_location"]].mean()
    adata.obs = adata.obs.join(centroids.rename(columns={"x_location": "x", "y_location": "y"}))
    adata.obsm["spatial"] = adata.obs[["x", "y"]].values
    return adata


adatas = {}
for method in METHODS:
    df = pd.read_parquet(TRANSCRIPT_FILES[method])
    cell_col = CELL_ID_COLS[method]
    adata = build_count_matrix(df, cell_col)
    adatas[method] = adata
    print(f"{method:10s} → {adata.n_obs:,} cells × {adata.n_vars} genes")

## 2. Standard Preprocessing Pipeline

Applied identically to all three methods.

In [ ]:
def preprocess(adata: ad.AnnData, method_name: str) -> ad.AnnData:
    adata = adata.copy()

    # QC — filter cells with very few transcripts
    sc.pp.filter_cells(adata, min_counts=10)
    sc.pp.filter_genes(adata, min_cells=5)

    # Normalise & log transform
    sc.pp.normalize_total(adata, target_sum=100)
    sc.pp.log1p(adata)

    # PCA → neighbours → UMAP → Leiden
    sc.pp.pca(adata, n_comps=30)
    sc.pp.neighbors(adata, n_neighbors=15, n_pcs=30)
    sc.tl.umap(adata)
    sc.tl.leiden(adata, resolution=0.5, key_added="leiden")

    print(f"{method_name}: {adata.n_obs:,} cells, {adata.obs['leiden'].nunique()} Leiden clusters")
    return adata


processed = {method: preprocess(adatas[method], method) for method in METHODS}

## 3. UMAP Visualisation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, method in zip(axes, METHODS):
    adata = processed[method]
    sc.pl.umap(
        adata,
        color="leiden",
        ax=ax,
        show=False,
        title=f"{method.capitalize()} ({adata.n_obs:,} cells)",
        legend_loc="on data",
        legend_fontsize=8,
    )
plt.tight_layout()
plt.savefig(RESULTS_DIR / "umap_leiden_all_methods.png", dpi=150)
plt.show()

## 4. Marker Gene Expression per Cluster

In [ ]:
MARKER_GENES = {
    "Epithelial": ["EPCAM", "KRT17", "KRT8"],
    "Macrophage": ["CD68", "CD163"],
    "T cell": ["CD3E", "CD8A"],
    "Fibroblast": ["COL1A1", "FAP"],
    "Endothelial": ["PECAM1", "VWF"],
}
all_markers = [g for genes in MARKER_GENES.values() for g in genes]

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
for ax, method in zip(axes, METHODS):
    adata = processed[method]
    available = [g for g in all_markers if g in adata.var_names]
    sc.pl.dotplot(
        adata,
        var_names=available,
        groupby="leiden",
        ax=ax,
        show=False,
        title=f"{method.capitalize()}",
        standard_scale="var",
    )
plt.tight_layout()
plt.savefig(RESULTS_DIR / "marker_dotplot_all_methods.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Cluster Proportion Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, method in zip(axes, METHODS):
    adata = processed[method]
    props = adata.obs["leiden"].value_counts(normalize=True).sort_index()
    ax.bar(props.index.astype(str), props.values, color=plt.cm.tab20.colors[:len(props)])
    ax.set_xlabel("Leiden cluster")
    ax.set_ylabel("Proportion")
    ax.set_title(f"{method.capitalize()} — Cluster Proportions")
    ax.tick_params(axis="x", labelsize=8)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "cluster_proportions_all_methods.png", dpi=150)
plt.show()

## 6. Spatial Visualisation of Cell Types

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, method in zip(axes, METHODS):
    adata = processed[method]
    sc.pl.spatial(
        adata,
        color="leiden",
        spot_size=15,
        ax=ax,
        show=False,
        title=f"{method.capitalize()} — Spatial",
        legend_loc=None,
    )
plt.tight_layout()
plt.savefig(RESULTS_DIR / "spatial_clusters_all_methods.png", dpi=150)
plt.show()

## 7. Spatial Neighbourhood Enrichment (Squidpy)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, method in zip(axes, METHODS):
    adata = processed[method]
    sq.gr.spatial_neighbors(adata, coord_type="generic", delaunay=True)
    sq.gr.nhood_enrichment(adata, cluster_key="leiden")
    sq.pl.nhood_enrichment(
        adata,
        cluster_key="leiden",
        ax=ax,
        show=False,
        title=f"{method.capitalize()} — Neighbourhood Enrichment",
    )
plt.tight_layout()
plt.savefig(RESULTS_DIR / "neighbourhood_enrichment_all_methods.png", dpi=150)
plt.show()

## 8. Save Processed AnnData Objects

In [ ]:
for method, adata in processed.items():
    out = DATA_DIR / f"adata_{method}_processed.h5ad"
    adata.write_h5ad(out)
    print(f"Saved {method} → {out}")